# 1D Drone Altitude Estimation: Multi-Sensor Fusion with Kalman Filters

**Objective:** Estimate the true altitude of a hovering drone by fusing data from two flawed sensors.

In the real world, whether you are trying to stabilize a quadcopter or extract clean end-effector coordinates for an imitation learning policy, sensors lie. Any control algorithm that assumes perfect state observations will quickly fail when exposed to real hardware noise. 

This notebook demonstrates a foundational concept in state estimation: **Sensor Fusion via a 1D Kalman Filter**. 

We will simulate a drone attempting to hold a steady 10-meter altitude and track its state using two imperfect data streams:
1. **A Barometer:** Measures air pressure. It provides smooth, continuous data but suffers from slow, unpredictable drift.
2. **An Ultrasonic Sensor:** Bounces sound waves off the ground. It is generally accurate but prone to massive, random spikes or dropouts.

By utilizing the predictable physics of the drone (the *Predict* step) and mathematically weighting the confidence of each sensor (the *Update* step), the Kalman Filter acts as a highly intelligent weighted average. It filters out the barometer's drift and ignores the ultrasonic spikes, outputting a clean, reliable state estimate that is actually more accurate than either individual sensor.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(42)


## Part 2: Initializing the Filter Matrices

Before the filter can run, we must define the shape of our world. 

**The Dimensions Rule:**
* **Variables (n) = 2:** We are tracking Altitude and Vertical Velocity.
* **Sensors (m) = 2:** We have a Barometer and an Ultrasonic sensor.

Because of this, our matrices will primarily be 2x1 vectors or 2x2 grids.

* **State (x):** A 2x1 column tracking [Altitude, Velocity].
* **Covariance (P):** A 2x2 grid. Why 2x2? Because it must track the uncertainty of Altitude (top-left), the uncertainty of Velocity (bottom-right), AND how Altitude and Velocity affect each other (the cross-correlations in the top-right and bottom-left).
* **Physics (F):** A 2x2 matrix that multiplies against the 2x1 state to calculate the new state.
* **Measurement Map (H):** A 2x2 matrix. It has 2 rows (one for each sensor) and 2 columns (one for each state variable). It tells the computer which state variable each sensor is looking at.

In [ ]:
dt=0.1 # time step : we run the filter 10 times a second

#initial state x, starting guess for the drone's pos and speed
#dimension (2,1)
x=np.array([[0.0], # initial altitude guess
            [0.0]]) # initial velocity guess

#initial covariance "doubt"
#how certain we are about our initial state x
#(2,2) # 100 on diagonal means we are highly uncertain
# [Variance of Altitude,      Correlation Alt-Vel]
# [Correlation Vel-Alt,       Variance of Velocity]
P=np.array([[100.0,0.0],
            [0.0,100.0]])

# 3. STATE TRANSITION MATRIX (F) -> Dimension: (2, 2)
# Equation: Altitude = 1*Altitude + dt*Velocity
# Equation: Velocity = 0*Altitude + 1*Velocity
F=np.array([[1.0,dt],
            [0.0,1.0]])


#4. MEASUREMENT MATRIX (H) -> Dimension: (2, 2)
# Row 1 (Barometer): [1.0, 0.0] -> Measures Altitude(1), Ignores Velocity(0)
# Row 2 (Ultrasonic): [1.0, 0.0] -> Measures Altitude(1), Ignores Velocity(0)
H=np.array([[1.0,0.0],
            [1.0,0.0]])

# 5. MEASUREMENT NOISE (R) -> Dimension: (2, 2)
# [Barometer Variance,     Correlation]
# [Correlation,            Ultrasonic Variance]
# Barometer drifts a bit (2.0), Ultrasonic is jittery (5.0)
R=np.array([[2.0,0.0],
            [0.0,5.0]])

# 6. PROCESS NOISE (Q) -> Dimension: (2, 2)
# Represents physical chaos (like wind). Kept very small.
Q = np.array([[0.1, 0.0], 
              [0.0, 0.1]])

# IDENTITY MATRIX (I) -> Dimension: (2, 2)
I = np.eye(2)